# Notebook 06 — Writing Faster Queries (Things YOU Control)
## Free Performance Wins — No Platform Team Needed

**What you'll learn**: These optimizations are 100% in YOUR hands as an analyst. No tickets, no waiting — just better SQL patterns that save time and money.

| | Good Habit | Bad Habit |
|---|---|---|
| **Columns** | Select only what you need | `SELECT *` (reads all 13 columns) |
| **Filters** | Filter early, before JOINs | Join everything first, filter later |
| **Caching** | Use fixed dates in dashboards | `CURRENT_TIMESTAMP()` defeats the cache |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Projection Pruning — `SELECT *` vs. Specific Columns

**Scenario**: Generate a monthly credit card statement (you only need date, merchant, amount).

### Worst Case: SELECT *

In [ ]:
-- WORST: Scans all 13 columns for 1 month of data
SELECT *
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
  AND account_id = 'ACC-0000000100';

### Best Case: Only Needed Columns

In [ ]:
-- BEST: Only 3 columns scanned
SELECT transaction_date, merchant_name, amount
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-01'
  AND transaction_date < '2024-12-01'
  AND account_id = 'ACC-0000000100';

In [ ]:
-- Compare bytes scanned
SELECT
    CASE WHEN query_text ILIKE '%SELECT *%' THEN 'SELECT *' ELSE 'SELECT cols' END AS approach,
    bytes_scanned / (1024*1024) AS mb_scanned,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -5, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 5
))
WHERE query_text ILIKE '%ACC-0000000100%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 2: Filter Placement — Before vs. After JOIN

**Scenario**: "Find high-value transactions for affluent customers in the last week."

### Worst Case: Join All Rows, Then Filter

In [ ]:
-- WORST: Joins 500M transactions to 25M accounts FIRST, then filters
SELECT t.transaction_date, t.merchant_name, t.amount, c.segment
FROM TRANSACTIONS t
JOIN ACCOUNTS a ON t.account_id = a.account_id
JOIN CUSTOMERS c ON a.customer_id = c.customer_id
WHERE c.segment = 'affluent'
  AND t.amount > 5000
  AND t.transaction_date >= DATEADD(day, -7, '2024-12-01'::TIMESTAMP_NTZ)
ORDER BY t.amount DESC
LIMIT 20;

### Best Case: Filter Early with CTEs

In [ ]:
-- BEST: Filter transactions down first, then join to small result set
WITH high_value_recent AS (
    SELECT account_id, transaction_date, merchant_name, amount
    FROM TRANSACTIONS
    WHERE transaction_date >= DATEADD(day, -7, '2024-12-01'::TIMESTAMP_NTZ)
      AND amount > 5000
)
SELECT h.transaction_date, h.merchant_name, h.amount, c.segment
FROM high_value_recent h
JOIN ACCOUNTS a ON h.account_id = a.account_id
JOIN CUSTOMERS c ON a.customer_id = c.customer_id
WHERE c.segment = 'affluent'
ORDER BY h.amount DESC
LIMIT 20;

**Note**: Snowflake's optimizer often rewrites these to be equivalent, but with complex queries and multiple joins, explicit early filtering helps the optimizer make better choices. Check the Query Profile to see the difference in rows processed at each stage.

---
## Step 3: Result Cache — Free Performance

**Scenario**: Branch manager dashboard — same "today's summary" query run every few minutes.

In [ ]:
-- First run: computes from scratch
SELECT
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_volume
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-30'
  AND transaction_date < '2024-12-01'
GROUP BY 1;

In [ ]:
-- Second run: IDENTICAL query hits result cache (sub-100ms)
SELECT
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_volume
FROM TRANSACTIONS
WHERE transaction_date >= '2024-11-30'
  AND transaction_date < '2024-12-01'
GROUP BY 1;

### Worst Case: Defeating the Cache

In [ ]:
-- WORST: Using CURRENT_TIMESTAMP() forces re-execution every time (cache miss)
SELECT
    channel,
    COUNT(*) AS txn_count,
    SUM(amount) AS total_volume
FROM TRANSACTIONS
WHERE transaction_date >= DATEADD(day, -1, CURRENT_TIMESTAMP())
GROUP BY 1;

---
## Key Takeaways — Your Personal Performance Checklist

| Habit | Why It Matters | Savings |
|-------|---------------|--------|
| **Never use `SELECT *` in reports** | You probably need 3-5 columns, not 13 | 2-4x less data scanned |
| **Filter before you JOIN** | Reduce 500M rows to 1M BEFORE joining | 10x+ fewer rows processed |
| **Use fixed dates in dashboards** | `'2024-12-01'` caches for 24 hours; `CURRENT_DATE()` re-computes every run | Free repeated results |

**These are FREE optimizations** — no extra features, no platform team requests. Just better SQL habits that save compute on every single query.

**Pro tip**: After writing any new report query, open the **Query Profile** and check:
1. How much data was scanned? (less = better)
2. How many rows went through the JOIN? (filter first = fewer rows)
3. Did it hit the result cache on re-run? (cache = free)

**Next** → [Notebook 07 — Diagnosing Slow Queries](./Notebook_07_Decision_Framework.ipynb)